In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import numpy as np
import pandas as pd
import os , re

In [2]:
reviews_train = []
for line in open(r'full_train.txt', 'r',encoding = 'utf-8'):

    reviews_train.append(line.strip().lower())

reviews_test = []
for line in open(r'full_test.txt', 'r',encoding = 'utf-8'):

    reviews_test.append(line.strip().lower())

In [3]:
reviews_train[5]

"this isn't the comedic robin williams, nor is it the quirky/insane robin williams of recent thriller fame. this is a hybrid of the classic drama without over-dramatization, mixed with robin's new love of the thriller. but this isn't a thriller, per se. this is more a mystery/suspense vehicle through which williams attempts to locate a sick boy and his keeper.<br /><br />also starring sandra oh and rory culkin, this suspense drama plays pretty much like a news report, until william's character gets close to achieving his goal.<br /><br />i must say that i was highly entertained, though this movie fails to teach, guide, inspect, or amuse. it felt more like i was watching a guy (williams), as he was actually performing the actions, from a third person perspective. in other words, it felt real, and i was able to subscribe to the premise of the story.<br /><br />all in all, it's worth a watch, though it's definitely not friday/saturday night fare.<br /><br />it rates a 7.7/10 from...<br />

| Regex | Matches            |
| ----- | ------------------ |
| `\.`  | .                  |
| `\;`  | ;                  |
| `\:`  | :                  |
| `\!`  | !                  |
| `\'`  | '                  |
| `\?`  | ?                  |
| `\,`  | ,                  |
| `\"`  | "                  |
| `\(`  | (                  |
| `\)`  | )                  |
| `\[`  | [                  |
| `\]`  | ]                  |
| `\d+` | one or more digits |


| Regex              | Meaning                        |
| ------------------ | ------------------------------ |
| `<br\s*/><br\s*/>` | HTML line break `<br /><br />` |
| `\-`               | hyphen `-`                     |
| `\/`               | slash `/`                      |


In [4]:
REPLACE_NO_SPACE = re.compile("(\.)|(\;)|(\:)|(\!)|(\')|(\?)|(\,)|(\")|(\()|(\))|(\[)|(\])|(\d+)")
REPLACE_WITH_SPACE = re.compile("(<br\s*/><br\s*/>)|(\-)|(\/)")
NO_SPACE = ""
SPACE = " "

def preprocess_reviews(reviews):

    reviews = [REPLACE_NO_SPACE.sub(NO_SPACE, line.lower()) for line in reviews]
    reviews = [REPLACE_WITH_SPACE.sub(SPACE, line) for line in reviews]

    return reviews

reviews_train_clean = preprocess_reviews(reviews_train)
reviews_test_clean = preprocess_reviews(reviews_test)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_4292/2185919213.py:1: SyntaxWarning: invalid escape sequence '\.'
  REPLACE_NO_SPACE = re.compile("(\.)|(\;)|(\:)|(\!)|(\')|(\?)|(\,)|(\")|(\()|(\))|(\[)|(\])|(\d+)")
/tmp/ipykernel_4292/2185919213.py:2: SyntaxWarning: invalid escape sequence '\s'
  REPLACE_WITH_SPACE = re.compile("(<br\s*/><br\s*/>)|(\-)|(\/)")


In [5]:
reviews_train_clean[5]

'this isnt the comedic robin williams nor is it the quirky insane robin williams of recent thriller fame this is a hybrid of the classic drama without over dramatization mixed with robins new love of the thriller but this isnt a thriller per se this is more a mystery suspense vehicle through which williams attempts to locate a sick boy and his keeper also starring sandra oh and rory culkin this suspense drama plays pretty much like a news report until williams character gets close to achieving his goal i must say that i was highly entertained though this movie fails to teach guide inspect or amuse it felt more like i was watching a guy williams as he was actually performing the actions from a third person perspective in other words it felt real and i was able to subscribe to the premise of the story all in all its worth a watch though its definitely not friday saturday night fare it rates a   from the fiend '

In [7]:
len(reviews_train_clean)

25000

In [8]:
len(reviews_test_clean)

25000

In [64]:
# cv = CountVectorizer(binary=True)
cv=TfidfVectorizer(min_df=2,max_df=0.9,use_idf=True)
cv.fit(reviews_train_clean)
X = cv.transform(reviews_train_clean)
X_test = cv.transform(reviews_test_clean)

In [66]:
cv.get_feature_names_out()[::-1][:5]

array(['über', 'émigré', 'élan', 'ème', 'zz'], dtype=object)

In [56]:
cv.vocabulary_

{'bromwell': 10215,
 'high': 36732,
 'is': 41093,
 'cartoon': 12200,
 'comedy': 15431,
 'it': 41273,
 'ran': 64688,
 'at': 4688,
 'the': 80175,
 'same': 69201,
 'time': 81101,
 'as': 4356,
 'some': 74449,
 'other': 57560,
 'programs': 63005,
 'about': 292,
 'school': 70141,
 'life': 46336,
 'such': 77527,
 'teachers': 79420,
 'my': 53987,
 'years': 89971,
 'in': 39341,
 'teaching': 79423,
 'profession': 62937,
 'lead': 45578,
 'me': 50321,
 'to': 81411,
 'believe': 7131,
 'that': 80116,
 'highs': 36758,
 'satire': 69556,
 'much': 53499,
 'closer': 14766,
 'reality': 65178,
 'than': 80087,
 'scramble': 70383,
 'survive': 78172,
 'financially': 29018,
 'insightful': 40227,
 'students': 77111,
 'who': 88283,
 'can': 11605,
 'see': 70833,
 'right': 67444,
 'through': 80840,
 'their': 80213,
 'pathetic': 59093,
 'pomp': 61616,
 'pettiness': 60123,
 'of': 56475,
 'whole': 88298,
 'situation': 73194,
 'all': 2102,
 'remind': 66270,
 'schools': 70165,
 'knew': 44091,
 'and': 2861,
 'when': 880

In [67]:
target = [1 if i < 12500 else 0 for i in range(25000)]

# [1,1,1,1,1      0,0,0,0,0,]

X_train, X_val, y_train, y_val = train_test_split(X, target, train_size = 0.75)

for c in [0.01, 0.05, 0.25, 0.5, 1]:
    lr = LogisticRegression(C=c,solver='lbfgs',max_iter=10000,penalty='l2',random_state=42)
    lr.fit(X_train, y_train)
    print ("Accuracy for C=%s: %s"% (c, accuracy_score(y_val, lr.predict(X_val))))

Accuracy for C=0.01: 0.81168
Accuracy for C=0.05: 0.84512
Accuracy for C=0.25: 0.8752
Accuracy for C=0.5: 0.88592
Accuracy for C=1: 0.89392


In [71]:
final_model = LogisticRegression(C=1,solver='lbfgs',max_iter=10000,penalty='l2',random_state=42)
final_model.fit(X, target)
print ("Final Accuracy: %s"% accuracy_score(target, final_model.predict(X_test)))
# Final Accuracy: 0.8858

Final Accuracy: 0.8858


In [72]:
print(final_model.predict(cv.transform(['not good'])))

[1]


In [73]:
print(final_model.coef_.shape)
print(final_model.intercept_.shape)

(1, 47093)
(1,)


In [74]:
feature_to_coef = {word: coef for word, coef in zip(cv.get_feature_names_out(), final_model.coef_[0])}

In [82]:
for word,best_positive in sorted(feature_to_coef.items(),key=lambda x: x[1],reverse=True)[:5]:
    print (word,">>>>",best_positive)

great >>>> 7.874108647379584
excellent >>>> 7.15415759276022
perfect >>>> 5.659534547213672
best >>>> 5.486840838945614
wonderful >>>> 5.197560683174949


In [83]:
for best_negative in sorted(feature_to_coef.items(),key=lambda x: x[1])[:5]:
    print (best_negative)

('worst', np.float64(-10.502883863049318))
('bad', np.float64(-8.119929142810925))
('awful', np.float64(-7.515626164975464))
('waste', np.float64(-7.397961750821382))
('boring', np.float64(-6.604390511520129))


In [76]:
print(f'Number of words values are {len(feature_to_coef)}')

Number of words values are 47093


In [77]:
newfile = open('SA Dictionary.txt', 'w',encoding = 'utf-8')

for w,v in feature_to_coef.items() :
    newfile.write(f'{w},{v}\n')

newfile.close()

In [78]:
with open('SA Dictionary.txt', 'r',encoding = 'utf-8') as f:
    for i,line in enumerate(f):
        print(i+1," ",line)

1   ____,-0.01861645832204504

2   ______,0.2938850527808071

3   _a,-0.0003567973538797493

4   _is_,-0.021649848294333383

5   _not_,0.0007777934600664805

6   _plan,-0.05900314262575312

7   _real_,-0.09279537038017893

8   _the,0.0990171818679931

9   aa,-0.18944037160649832

10   aaa,-0.16970032282645842

11   aaargh,-0.03511549138043893

12   aag,-0.22497571789325077

13   aage,0.06720085419620354

14   aaker,0.03942435023748525

15   aames,0.17867307688400222

16   aamir,-0.0755288216873661

17   aan,0.053455095866716734

18   aankhen,0.054682268440911695

19   aapke,-0.1292933832548267

20   aardman,-0.11425195298773777

21   aardmans,-0.020709627194750432

22   aargh,-0.05295179391095774

23   aaron,-0.1944165177290044

24   aarons,0.05869842183005529

25   aarp,-0.06041782706509966

26   aatish,0.13094845016342882

27   aavjo,0.03401112243394947

28   ab,0.1442347397530842

29   aback,0.04583971360218385

30   abadi,0.040357594248263365

31   abandon,-0.0672398284573356

32  